# Mistral confirmatory factorial v1 — frozen 512-row run

This notebook executes the confirmatory token identity × positional
geometry experiment only after verifying the exact passed Mistral
qualification audit. The frozen grid contains 32 fresh semantic seeds × 2
filler languages × 2 target lengths × 4 conditions = 512 rows.

Conditions are randomized within each semantic instance. Results are
append-only and resume-safe. A result file may be resumed only under the
identical runtime fingerprint. No effect estimate is computed until all
512 rows and all pairing/geometry gates pass.

The adapter retains Mistral byte-fallback tokens canonically and uses the
token-offset evidence/filler/query labels frozen before task scoring.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import json, shutil, subprocess, sys

PROJECT_DIR = Path('/content/drive/MyDrive/tacl_factorization')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
HARNESS_ZIP = PROJECT_DIR / 'TACL_Expansion_Harness_v4_5.zip'
REPO_DIR = Path('/content/causal_factorization')

if not HARNESS_ZIP.exists():
    print('Upload TACL_Expansion_Harness_v4_5.zip once; it will be retained in Drive.')
    uploaded = files.upload()
    names = [name for name in uploaded if name.endswith('.zip')]
    if not names:
        raise RuntimeError('No harness ZIP was uploaded.')
    shutil.copy2(names[0], HARNESS_ZIP)

if not REPO_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/bilkentli88/causal_factorization', str(REPO_DIR)
    ])

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    str(HARNESS_ZIP), str(REPO_DIR / 'code/fragburden0.3.0.zip')
])
# A previous notebook may have imported an older harness in this same runtime.
# Evict only this package so subsequent imports use the just-installed version.
for module_name in list(sys.modules):
    if module_name == 'tacl_factorization' or module_name.startswith('tacl_factorization.'):
        del sys.modules[module_name]
import importlib
importlib.invalidate_caches()
print('Setup complete.')

In [ ]:
import hashlib
from tacl_factorization.config import assert_frozen, config_hash, load_config

EXTRACTED = Path('/content/tacl_harness_files')
if EXTRACTED.exists(): shutil.rmtree(EXTRACTED)
shutil.unpack_archive(str(HARNESS_ZIP), str(EXTRACTED))
config_candidates = list(EXTRACTED.rglob('mistral_factorial_confirmatory_v1.json'))
if len(config_candidates) != 1: raise RuntimeError('Could not locate frozen Mistral factorial config.')
cfg = load_config(config_candidates[0])
assert_frozen(cfg)
print('Frozen factorial config hash:', config_hash(cfg))
print(json.dumps(cfg, indent=2))

qualification_path = PROJECT_DIR / cfg['qualification']['audit_file']
if not qualification_path.exists():
    raise RuntimeError('Run the Mistral engineering qualification notebook first.')
qualification_bytes = qualification_path.read_bytes()
qualification_sha = hashlib.sha256(qualification_bytes).hexdigest()
qualification = json.loads(qualification_bytes)
assert qualification_sha == cfg['qualification']['audit_sha256'], 'Qualification audit hash differs.'
assert qualification['config_hash'] == cfg['qualification']['qualification_config_hash']
assert qualification['passed'] is True
assert qualification['factorial_outcomes_accessed'] is False
assert qualification['selected_keep_by_language'] == cfg['frag_keep_by_language']
print('PASS: exact qualification audit verified.')

import tacl_factorization
assert tacl_factorization.__version__ == '0.4.5'
assert qualification['static_gate']['region_labeling'] == cfg['region_labeling']
assert qualification['static_gate']['byte_fallback_policy'] == cfg['byte_fallback_policy']
assert qualification['static_gate']['shell_extraction'] == cfg['chat_shell_extraction']
assert qualification['static_gate']['digit_ids'] == cfg['candidate_token_ids']
print('PASS: Mistral tokenizer and readout corrections are audit-bound.')


In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.15.1', 'accelerate', 'bitsandbytes', 'tokenizers'
])

import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from fragburden.dial import FragmentationDial, verify_monotone
from fragburden.testing import sample_texts
from tacl_factorization.adapters import HFChatAdapter
from tacl_factorization.results import read_jsonl

assert torch.cuda.is_available(), 'Select a GPU runtime.'
assert torch.cuda.is_bf16_supported(), 'The frozen BF16 run requires a BF16-capable GPU.'
assert transformers.__version__ == cfg['transformers_version']
tokenizer = AutoTokenizer.from_pretrained(cfg['model'], use_fast=True)
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
load_kwargs = dict(
    quantization_config=bnb, device_map='auto', attn_implementation='sdpa'
)
try:
    model = AutoModelForCausalLM.from_pretrained(
        cfg['model'], dtype=torch.bfloat16, **load_kwargs
    ).eval()
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        cfg['model'], torch_dtype=torch.bfloat16, **load_kwargs
    ).eval()
if hasattr(model.config, 'use_cache'): model.config.use_cache = False

runtime = {
    'model': cfg['model'], 'weights': cfg['weights'],
    'harness_version': tacl_factorization.__version__,
    'region_labeling': cfg['region_labeling'],
    'byte_fallback_policy': cfg['byte_fallback_policy'],
    'compute_dtype': cfg['compute_dtype'],
    'transformers': transformers.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
}
runtime_fingerprint = hashlib.sha256(
    json.dumps(runtime, sort_keys=True, separators=(',', ':')).encode()
).hexdigest()[:24]
print(json.dumps(runtime, indent=2))
print('Runtime fingerprint:', runtime_fingerprint)

In [ ]:
dial = FragmentationDial.from_transformers(tokenizer, name=cfg['model'])
dial_gate = verify_monotone(
    dial, sample_texts('en') + sample_texts('tr'),
    levels=(100, *sorted(set(cfg['frag_keep_by_language'].values()), reverse=True))
)
assert dial_gate['monotone_nondecreasing'] and dial_gate['all_in_vocab']
adapter = HFChatAdapter(
    tokenizer=tokenizer,
    base_dial=dial.at(100),
    fragmented_dials={
        language: dial.at(cfg['frag_keep_by_language'][language])
        for language in cfg['filler_languages']
    },
    chat_kwargs=cfg['chat_kwargs'],
    answer_prefix_ids=cfg['answer_prefix_ids'],
    candidate_token_ids=cfg['candidate_token_ids'],
)
assert adapter.digit_ids == qualification['static_gate']['digit_ids']
print('Factorial adapter ready. keep_pct:', cfg['frag_keep_by_language'])

from tacl_factorization.generator import build_instance
from tacl_factorization.geometry import assemble_factorial

probe_instance = build_instance(
    filler_language=cfg['filler_languages'][0],
    semantic_seed=cfg['semantic_seeds'][0],
    trial_index=0,
    target_chat_tokens=cfg['target_lengths'][0],
    count_chat_tokens=adapter.count_chat_tokens,
    verify_chat_shell=adapter.verify_chat_shell,
    n_records=cfg['n_records'],
)
probe_encoding = adapter.encode_instance(probe_instance)
assert adapter.last_encoding_strategy == cfg['region_labeling']
probe_factorial = assemble_factorial(
    adapter.prefix_ids, adapter.suffix_ids, probe_encoding
)
assert probe_factorial['CC'].ids_hash == probe_factorial['CE'].ids_hash
assert probe_factorial['FC'].ids_hash == probe_factorial['FE'].ids_hash
assert probe_factorial['CC'].gaps['evidence_readout'] == probe_factorial['FC'].gaps['evidence_readout']
assert probe_factorial['CE'].gaps['evidence_readout'] == probe_factorial['FE'].gaps['evidence_readout']
print('PASS: pre-outcome Mistral adapter and geometry probe.')


In [ ]:
RESULTS_PATH = PROJECT_DIR / 'mistral_factorial_confirmatory_v1.jsonl'
existing_rows = read_jsonl(RESULTS_PATH)
if existing_rows:
    fingerprints = {row.get('runtime_fingerprint') for row in existing_rows}
    if fingerprints != {runtime_fingerprint}:
        raise RuntimeError(
            'Existing rows were produced under another runtime. Restore that runtime '
            'or start a separately named result file; never mix them.'
        )
print('Existing rows:', len(existing_rows), '/', cfg['expected_rows'])

RUN_CONFIRMATORY_FACTORIAL = False  # Read the frozen config above, then change only this to True.
if not RUN_CONFIRMATORY_FACTORIAL:
    print('Confirmatory run is armed but not started. Set RUN_CONFIRMATORY_FACTORIAL=True.')

In [ ]:
if RUN_CONFIRMATORY_FACTORIAL:
    from tacl_factorization.experiment import run_symbolic_factorial
    from tacl_factorization.scoring import score_one_shot

    provenance = {
        'factorial_config_hash': config_hash(cfg),
        'qualification_audit_sha256': qualification_sha,
        'runtime_fingerprint': runtime_fingerprint,
        'runtime': runtime,
    }
    new_rows = run_symbolic_factorial(
        config=cfg,
        adapter=adapter,
        model=model,
        output_path=RESULTS_PATH,
        scorer=score_one_shot,
        limit_new_rows=None,
        row_provenance=provenance,
    )
    print('New rows written:', new_rows)
    print('Result file:', RESULTS_PATH)

## Locked analysis gate

This cell refuses to calculate effects from a partial or structurally
invalid run. Run it only after the execution cell finishes.

In [ ]:
from tacl_factorization.statistics import (
    canonical_metrics, factorial_effects, load_jsonl, summarize_factorial,
    validate_factorial_rows,
)
import pandas as pd

rows = load_jsonl(RESULTS_PATH)
print('Rows present:', len(rows), '/', cfg['expected_rows'])
if len(rows) != cfg['expected_rows']:
    print('INCOMPLETE: effect analysis remains locked.')
else:
    validate_factorial_rows(rows)
    assert set(rows['runtime_fingerprint']) == {runtime_fingerprint}
    base = ['filler_language', 'target_chat_tokens', 'semantic_seed', 'trial_index']
    assert (rows.groupby(base)['prompt_sha256'].nunique() == 1).all()
    pair = rows.pivot_table(
        index=base, columns='factorial_code',
        values=['ids_hash', 'evidence_readout_position_gap'], aggfunc='first'
    )
    assert (pair['ids_hash']['CC'] == pair['ids_hash']['CE']).all()
    assert (pair['ids_hash']['FC'] == pair['ids_hash']['FE']).all()
    assert ((pair['evidence_readout_position_gap']['CC'] - pair['evidence_readout_position_gap']['FC']).abs() <= 1e-3).all()
    assert ((pair['evidence_readout_position_gap']['CE'] - pair['evidence_readout_position_gap']['FE']).abs() <= 1e-3).all()
    assert (pair['evidence_readout_position_gap']['FE'] > pair['evidence_readout_position_gap']['FC']).all()
    print('PASS: completeness, same-ID, matched-geometry, prompt, and runtime gates.')

    pooled = summarize_factorial(
        rows, bootstrap_seed=cfg['bootstrap']['seed'], reps=cfg['bootstrap']['reps']
    )
    cells = {}
    for (language, length), group in rows.groupby(['filler_language', 'target_chat_tokens']):
        cells[f'{language}_{int(length)}'] = summarize_factorial(
            group, bootstrap_seed=cfg['bootstrap']['seed'], reps=cfg['bootstrap']['reps']
        )
    summary = {
        'protocol_version': cfg['protocol_version'],
        'factorial_config_hash': config_hash(cfg),
        'qualification_audit_sha256': qualification_sha,
        'runtime_fingerprint': runtime_fingerprint,
        'rows': len(rows),
        'canonical_metrics': canonical_metrics(rows),
        'pooled_effects': pooled,
        'cell_effects': cells,
    }
    SUMMARY_PATH = PROJECT_DIR / 'mistral_factorial_confirmatory_v1_summary.json'
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    display(pd.DataFrame(pooled).T)
    print('Frozen summary:', SUMMARY_PATH)